# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The purpose of this rule is to identify pages that appear to underperform relative to their visibility.

The rule relies on two signals:

- High impressions combined with low CTR.
- Older content with sufficient search demand.

Each page receives a score, a reason code, and an action label.

Possible reason codes:

- HIGH_IMPRESSIONS_LOW_CTR
- HIGH_VOLUME_OLD_CONTENT
- LOW_ENGAGEMENT

Possible actions:

- Review title and metadata.
- Improve content structure.
- Monitor performance.

In [23]:
!git clone https://github.com/ch-jsk/jsk-flyrank-ml-internship.git

Cloning into 'jsk-flyrank-ml-internship'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 142 (delta 53), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.86 MiB | 13.68 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [24]:
%cd jsk-flyrank-ml-internship

/content/jsk-flyrank-ml-internship/jsk-flyrank-ml-internship


In [25]:
!ls data/raw

content_refresh_anonymized.csv


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

df[["impressions_90d", "ctr", "content_age_days"]].describe()

(30000, 44)


,impressions_90d,ctr,content_age_days
count,30000.000000,30000.000000,30000.00000
mean,5200.366300,0.510733,256.16780
std,16838.019547,3.279162,132.70793
min,1.000000,0.000000,90.00000
25%,81.000000,0.000000,132.00000
50%,731.000000,0.070000,236.00000
75%,3615.250000,0.290000,333.00000
max,517715.000000,100.000000,564.00000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [27]:
#CTR versus position:

bucket = pd.cut(
    df["avg_position"],
    bins=[0, 5, 10, 20, 50, 100],
)

signal_1 = (
    df.groupby(bucket)["ctr"]
      .agg(["mean", "count"])
)

signal_1

/tmp/ipykernel_3650/3135312612.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(bucket)["ctr"]


,mean,count
avg_position,,
"(0, 5]",1.572937,3923
"(5, 10]",0.511708,9060
"(10, 20]",0.323443,7273
"(20, 50]",0.222345,7225
"(50, 100]",0.152525,1299


VERDICT: CONFIRMED

In [28]:
#Content age versus trend:

bucket = pd.cut(
    df["content_age_days"],
    bins=[0, 180, 365, 730, 2000]
)

signal_2 = (
    df.groupby(bucket)["trend_pct"]
      .agg(["mean", "count"])
)

signal_2

/tmp/ipykernel_3650/813550756.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(bucket)["trend_pct"]


,mean,count
content_age_days,,
"(0, 180]",-1.997824,10891
"(180, 365]",-12.322987,9614
"(365, 730]",2.106976,6107
"(730, 2000]",NaN,0


VERDICT: MIXED

In [29]:
#build the score
df["baseline_score"] = (
    df["impressions_90d"].fillna(0) * 0.4
    + (1 - df["ctr"].fillna(0)) * 100 * 0.4
    + (df["content_age_days"].fillna(0) / 365) * 0.2
)

In [30]:
#reason code
df["reason_code"] = "MONITOR"

df.loc[
    (df["impressions_90d"] > 1000) &
    (df["ctr"] < 0.5),
    "reason_code"
] = "HIGH_IMPRESSIONS_LOW_CTR"

In [31]:
#action labels
df["action"] = "Monitor"

df.loc[
    df["reason_code"] == "HIGH_IMPRESSIONS_LOW_CTR",
    "action"
] = "Review metadata"

In [32]:
#create queue
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

queue.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
6653,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.23,26.90,0.38,excellent,page_1,down,-44.8,207120.694247,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
17812,content_aaef01a50def,client_19581e27de,4400.0,0.05,LOW,0.11,keyword article,informational,NaN,NaN,...,2.42,4.77,0.00,excellent,page_1,stable,-4.0,206873.843836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
26844,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,11.73,16.03,0.00,excellent,top_3,down,-44.5,203735.043836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
19636,content_2cb567c3c89b,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6183.0,39587.0,...,5.82,5.64,1.21,excellent,page_3_5,up,23.1,199126.883836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
21819,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,11.31,18.07,0.00,excellent,top_3,down,-33.2,185265.043836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata


In [33]:
import os
print(os.getcwd())

/content/jsk-flyrank-ml-internship/jsk-flyrank-ml-internship


In [34]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [35]:
!ls work

capstone_report_template.md  notebooks	README.md


In [36]:
import os

os.makedirs("work/outputs", exist_ok=True)

In [37]:
#save csv
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [38]:
top20 = queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20)

top20

,content_id,baseline_score,reason_code,action
6653,content_5fe46e04994d,207120.694247,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
17812,content_aaef01a50def,206873.843836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
26844,content_8c19996aa890,203735.043836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
19636,content_2cb567c3c89b,199126.883836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
21819,content_4c36c775b818,185265.043836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
29400,content_2dba2b1f9536,177405.363836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
29879,content_1a9e894be2e2,166503.064110,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
13537,content_2c2606c5d176,138978.598356,MONITOR,Monitor
18870,content_db5989a78dd3,138076.243836,HIGH_IMPRESSIONS_LOW_CTR,Review metadata
14090,content_44e481c8f55b,125091.661370,MONITOR,Monitor


### Row 1
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? The page may target highly competitive search terms.

### Row 2
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? Seasonal effects may have reduced click-through rates.

### Row 3
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? Search intent may not match the page content.

### Row 4
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? The page may already be performing well for its position tier.

### Row 5
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? The page could be affected by temporary ranking fluctuations.

### Row 6
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? The page may have insufficient engagement data.

### Row 7
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? External events may have affected user behaviour.

### Row 8
Action: Monitor.
Reason: The page received a lower baseline score.
What would make this wrong? More recent data could change the ranking.

### Row 9
Action: Review metadata.
Reason: High impressions and low CTR.
What would make this wrong? Search intent may already be correctly matched.

### Row 10
Action: Monitor.
Reason: The page requires additional observation.
What would make this wrong? A longer time window may reveal stronger signals.

### Row 11
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? Low engagement may be caused by external factors.

### Row 12
Action: Monitor.
Reason: The score suggests observation rather than intervention.
What would make this wrong? The page may improve naturally over time.

### Row 13
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? Search demand may have changed recently.

### Row 14
Action: Monitor.
Reason: The page requires further monitoring.
What would make this wrong? Additional engagement data may change the recommendation.

### Row 15
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? Ranking position may be the dominant factor.

### Row 16
Action: Monitor.
Reason: Moderate baseline score.
What would make this wrong? Future data may reveal stronger trends.

### Row 17
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? User intent could be different from the expected intent.

### Row 18
Action: Monitor.
Reason: The page requires additional analysis.
What would make this wrong? The available data may be incomplete.

### Row 19
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? Strong competition may explain the lower CTR.

### Row 20
Action: Review metadata.
Reason: High impressions and relatively low CTR.
What would make this wrong? Changes in ranking position could alter the result.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some pages may appear to be poor performers simply because they occupy lower search positions.

No future information, trend labels, or label-derived features were included in the rule.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.